# EduGuard AI — Final V2
**Graduate = 0 | Dropout = 1 | Enrolled scored after training | Prediction point = End of Semester 1**


In [ ]:
# 1) Imports
import json, joblib, warnings, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score, brier_score_loss,
    classification_report, ConfusionMatrixDisplay,
    precision_recall_curve, roc_curve
)
from xgboost import XGBClassifier


In [ ]:
# 2) Load Data
paths = [
    Path("../data/raw/data.csv"),
    Path("data/raw/data.csv"),
    Path("../data/raw/data(1).csv"),
    Path("data/raw/data(1).csv"),
    Path("../data(1).csv"),
    Path("data(1).csv")
]

DATA_PATH = next((p for p in paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("data/raw/data.csv not found")

df = pd.read_csv(DATA_PATH, sep=";")
df.columns = df.columns.str.strip()

print("Shape:", df.shape)
display(df.head())


In [ ]:
# 3) Data Quality + Target
print("Missing:", int(df.isna().sum().sum()))
print("Duplicates:", int(df.duplicated().sum()))
display(df["Target"].value_counts().to_frame("Count"))

df = df.drop_duplicates().reset_index(drop=True)

resolved = df[df["Target"].isin(["Graduate", "Dropout"])].copy()
enrolled = df[df["Target"] == "Enrolled"].copy()
resolved["Dropout"] = (resolved["Target"] == "Dropout").astype(int)

print("Training rows:", len(resolved))
print("Enrolled rows:", len(enrolled))


In [ ]:
# 4) EDA
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

df["Target"].value_counts().plot(kind="bar", ax=ax[0], title="Outcome Distribution")
ax[0].tick_params(axis="x", rotation=0)

resolved.boxplot(column="Curricular units 1st sem (approved)", by="Target", ax=ax[1])
ax[1].set_title("Approved Units")
ax[1].set_xlabel("")

resolved.boxplot(column="Curricular units 1st sem (grade)", by="Target", ax=ax[2])
ax[2].set_title("Semester-1 Grade")
ax[2].set_xlabel("")

plt.suptitle("")
plt.tight_layout()
plt.show()


In [ ]:
# 5) Leakage + Feature Types
second_sem_cols = [c for c in resolved.columns if "2nd sem" in c]

X = resolved.drop(columns=["Target", "Dropout"] + second_sem_cols)
y = resolved["Dropout"]

categorical_cols = [
    "Marital status", "Application mode", "Course", "Daytime/evening attendance",
    "Previous qualification", "Nacionality", "Mother's qualification",
    "Father's qualification", "Mother's occupation", "Father's occupation",
    "Displaced", "Educational special needs", "Debtor",
    "Tuition fees up to date", "Gender", "Scholarship holder", "International"
]
categorical_cols = [c for c in categorical_cols if c in X.columns]
numeric_cols = [c for c in X.columns if c not in categorical_cols]

print("Removed Semester-2 leakage:", len(second_sem_cols))
print("Features:", X.shape[1], "| Numeric:", len(numeric_cols), "| Categorical:", len(categorical_cols))


In [ ]:
# 6) Invalid Values + Outliers
VALID_RANGES = {
    "Application order": (0, 9),
    "Previous qualification (grade)": (0, 200),
    "Admission grade": (0, 200),
    "Age at enrollment": (15, 100),
    "Curricular units 1st sem (credited)": (0, np.inf),
    "Curricular units 1st sem (enrolled)": (0, np.inf),
    "Curricular units 1st sem (evaluations)": (0, np.inf),
    "Curricular units 1st sem (approved)": (0, np.inf),
    "Curricular units 1st sem (grade)": (0, 20),
    "Curricular units 1st sem (without evaluations)": (0, np.inf)
}

def clean_invalid_values(data):
    data = data.copy()
    invalid = {}
    for col, (low, high) in VALID_RANGES.items():
        if col not in data.columns:
            continue
        mask = (data[col] < low) | (data[col] > high)
        invalid[col] = int(mask.sum())
        data.loc[mask, col] = np.nan
    return data, invalid

X, invalid_train = clean_invalid_values(X)

outliers = []
for col in numeric_cols:
    q1, q3 = X[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = int(((X[col] < low) | (X[col] > high)).sum())
    outliers.append([col, count])

print("Invalid values:", invalid_train)
display(
    pd.DataFrame(outliers, columns=["Feature", "IQR Outliers"])
    .sort_values("IQR Outliers", ascending=False)
)

print("Valid extreme values are kept.")


In [ ]:
# 7) Correlation + Feature Screening
corr_target = (
    pd.concat([X[numeric_cols], y], axis=1)
    .corr(method="spearman")["Dropout"]
    .drop("Dropout")
    .sort_values(key=np.abs, ascending=False)
)
display(corr_target.to_frame("Spearman r").head(12))

corr = X[numeric_cols].corr(method="spearman").abs()
pairs = [
    (a, b, corr.loc[a, b])
    for i, a in enumerate(corr.columns)
    for b in corr.columns[i + 1:]
    if corr.loc[a, b] >= 0.90
]
display(pd.DataFrame(pairs, columns=["Feature A", "Feature B", "|r|"]))

zero_variance = [c for c in X.columns if X[c].nunique(dropna=False) <= 1]
if zero_variance:
    X = X.drop(columns=zero_variance)
    numeric_cols = [c for c in numeric_cols if c not in zero_variance]
    categorical_cols = [c for c in categorical_cols if c not in zero_variance]

print("Removed zero-variance features:", zero_variance)
print("Selected features:", X.shape[1])


In [ ]:
# 8) Split
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

X_train, X_cal, y_train, y_cal = train_test_split(
    X_dev, y_dev,
    test_size=0.20,
    stratify=y_dev,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape)
print("Calibration:", X_cal.shape)
print("Test:", X_test.shape)


In [ ]:
# 9) Preprocessing
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])

num_logistic = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

num_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

prep_logistic = ColumnTransformer([
    ("num", num_logistic, numeric_cols),
    ("cat", cat_pipe, categorical_cols)
])

prep_tree = ColumnTransformer([
    ("num", num_tree, numeric_cols),
    ("cat", cat_pipe, categorical_cols)
])


In [ ]:
# 10) Imbalance + Models
positive = int(y_train.sum())
negative = int((1 - y_train).sum())
scale_pos_weight = negative / positive

models = {
    "Logistic Regression": (
        prep_logistic,
        LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        )
    ),
    "Random Forest": (
        prep_tree,
        RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    ),
    "Extra Trees": (
        prep_tree,
        ExtraTreesClassifier(
            n_estimators=500,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    ),
    "XGBoost": (
        prep_tree,
        XGBClassifier(
            n_estimators=450,
            max_depth=4,
            learning_rate=0.03,
            subsample=0.85,
            colsample_bytree=0.85,
            min_child_weight=3,
            reg_lambda=2.0,
            scale_pos_weight=scale_pos_weight,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )
}

print("Graduate:", negative, "| Dropout:", positive)
print("scale_pos_weight:", round(scale_pos_weight, 3))


In [ ]:
# 11) 5-Fold CV + Overfitting
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    "PR_AUC": "average_precision",
    "ROC_AUC": "roc_auc",
    "Recall": "recall",
    "Precision": "precision",
    "F1": "f1"
}

rows = []
pipelines = {}

for name, (prep, clf) in models.items():
    pipe = Pipeline([
        ("preprocessor", clone(prep)),
        ("classifier", clf)
    ])

    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=True
    )

    train_pr = scores["train_PR_AUC"].mean()
    cv_pr = scores["test_PR_AUC"].mean()

    rows.append({
        "Model": name,
        "Train PR-AUC": train_pr,
        "CV PR-AUC": cv_pr,
        "Gap": train_pr - cv_pr,
        "CV ROC-AUC": scores["test_ROC_AUC"].mean(),
        "CV Recall": scores["test_Recall"].mean(),
        "CV Precision": scores["test_Precision"].mean(),
        "CV F1": scores["test_F1"].mean()
    })

    pipelines[name] = pipe

cv_results = (
    pd.DataFrame(rows)
    .sort_values("CV PR-AUC", ascending=False)
    .reset_index(drop=True)
)

display(cv_results)


In [ ]:
# 12) Best Model
best_model_name = cv_results.iloc[0]["Model"]
best_model = pipelines[best_model_name]
best_model.fit(X_train, y_train)

selected_gap = float(cv_results.iloc[0]["Gap"])

print("Selected:", best_model_name)
print("Train-CV PR-AUC gap:", round(selected_gap, 4))


In [ ]:
# 13) Calibration + Threshold
raw_cal_prob = best_model.predict_proba(X_cal)[:, 1]

calibrator = IsotonicRegression(out_of_bounds="clip")
calibrator.fit(raw_cal_prob, y_cal)
cal_prob = calibrator.predict(raw_cal_prob)

def choose_threshold(y_true, prob, minimum_recall=0.85):
    precision, recall, thresholds = precision_recall_curve(y_true, prob)
    candidates = []

    for i, t in enumerate(thresholds):
        if recall[i] >= minimum_recall:
            f1 = 2 * precision[i] * recall[i] / (precision[i] + recall[i] + 1e-12)
            candidates.append((t, precision[i], recall[i], f1))

    if candidates:
        return max(candidates, key=lambda x: (x[1], x[3]))

    f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    i = int(np.argmax(f1))
    return thresholds[i], precision[i], recall[i], f1[i]

threshold, cal_precision, cal_recall, cal_f1 = choose_threshold(y_cal, cal_prob)

print("Brier:", round(brier_score_loss(y_cal, cal_prob), 4))
print("Threshold:", round(float(threshold), 6))
print("Precision:", round(float(cal_precision), 4))
print("Recall:", round(float(cal_recall), 4))
print("F1:", round(float(cal_f1), 4))


In [ ]:
# 14) Final Test
raw_test_prob = best_model.predict_proba(X_test)[:, 1]
test_prob = calibrator.predict(raw_test_prob)
test_pred = (test_prob >= threshold).astype(int)

test_metrics = {
    "Accuracy": accuracy_score(y_test, test_pred),
    "Precision": precision_score(y_test, test_pred),
    "Recall": recall_score(y_test, test_pred),
    "F1": f1_score(y_test, test_pred),
    "PR-AUC": average_precision_score(y_test, test_prob),
    "ROC-AUC": roc_auc_score(y_test, test_prob),
    "Brier Score": brier_score_loss(y_test, test_prob)
}

display(pd.DataFrame([test_metrics]))
print(classification_report(
    y_test,
    test_pred,
    target_names=["Graduate", "Dropout"],
    digits=4
))


In [ ]:
# 15) Evaluation Plots
fig, ax = plt.subplots(2, 2, figsize=(12, 9))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_pred,
    display_labels=["Graduate", "Dropout"],
    ax=ax[0, 0]
)
ax[0, 0].set_title("Confusion Matrix")

fpr, tpr, _ = roc_curve(y_test, test_prob)
ax[0, 1].plot(fpr, tpr, label=f"AUC={test_metrics['ROC-AUC']:.3f}")
ax[0, 1].plot([0, 1], [0, 1], "--")
ax[0, 1].set_title("ROC Curve")
ax[0, 1].legend()

p, r, _ = precision_recall_curve(y_test, test_prob)
ax[1, 0].plot(r, p, label=f"PR-AUC={test_metrics['PR-AUC']:.3f}")
ax[1, 0].set_title("Precision-Recall Curve")
ax[1, 0].legend()

frac_pos, mean_pred = calibration_curve(
    y_test,
    test_prob,
    n_bins=8,
    strategy="quantile"
)
ax[1, 1].plot(mean_pred, frac_pos, marker="o", label="Model")
ax[1, 1].plot([0, 1], [0, 1], "--", label="Perfect")
ax[1, 1].set_title("Calibration Curve")
ax[1, 1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# 16) 95% Uncertainty
def bootstrap_metrics(y_true, prob, threshold, iterations=1000):
    rng = np.random.default_rng(RANDOM_STATE)
    y_arr = np.asarray(y_true)
    p_arr = np.asarray(prob)

    values = {
        "PR-AUC": [],
        "ROC-AUC": [],
        "Recall": [],
        "Precision": [],
        "F1": []
    }

    for _ in range(iterations):
        idx = rng.integers(0, len(y_arr), len(y_arr))
        ys, ps = y_arr[idx], p_arr[idx]

        if len(np.unique(ys)) < 2:
            continue

        pred = (ps >= threshold).astype(int)

        values["PR-AUC"].append(average_precision_score(ys, ps))
        values["ROC-AUC"].append(roc_auc_score(ys, ps))
        values["Recall"].append(recall_score(ys, pred))
        values["Precision"].append(precision_score(ys, pred, zero_division=0))
        values["F1"].append(f1_score(ys, pred))

    rows = []
    for metric, vals in values.items():
        low, median, high = np.percentile(vals, [2.5, 50, 97.5])
        rows.append([metric, median, low, high])

    return pd.DataFrame(
        rows,
        columns=["Metric", "Estimate", "95% CI Low", "95% CI High"]
    )

confidence_intervals = bootstrap_metrics(
    y_test,
    test_prob,
    threshold
)

display(confidence_intervals)


In [ ]:
# 17) Feature Importance
importance = permutation_importance(
    best_model,
    X_cal,
    y_cal,
    scoring="average_precision",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "Feature": X_cal.columns,
    "Importance": importance.importances_mean
}).sort_values("Importance", ascending=False)

display(importance_df.head(15))

importance_df.head(15).sort_values("Importance").plot(
    x="Feature",
    y="Importance",
    kind="barh",
    figsize=(8, 6),
    legend=False,
    title="Feature Importance"
)

plt.tight_layout()
plt.show()


In [ ]:
# 18) Risk Score + Current Enrolled Students
MEDIUM_RISK = 0.30
HIGH_RISK = float(threshold)
REVIEW_MARGIN = 0.08

def predict_student_risk(student_df):
    raw = best_model.predict_proba(student_df)[:, 1]
    prob = float(calibrator.predict(raw)[0])

    level = (
        "High" if prob >= HIGH_RISK
        else "Medium" if prob >= MEDIUM_RISK
        else "Low"
    )

    return {
        "risk_score": round(prob * 100, 2),
        "risk_level": level,
        "needs_human_review": abs(prob - HIGH_RISK) <= REVIEW_MARGIN
    }

enrolled_X = enrolled.drop(
    columns=["Target"] + second_sem_cols,
    errors="ignore"
)[X.columns]

enrolled_X, invalid_enrolled = clean_invalid_values(enrolled_X)

enrolled_prob = calibrator.predict(
    best_model.predict_proba(enrolled_X)[:, 1]
)

enrolled_scores = enrolled.copy()
enrolled_scores["Risk Score"] = np.round(enrolled_prob * 100, 2)
enrolled_scores["Risk Level"] = np.where(
    enrolled_prob >= HIGH_RISK,
    "High",
    np.where(enrolled_prob >= MEDIUM_RISK, "Medium", "Low")
)
enrolled_scores["Human Review"] = (
    np.abs(enrolled_prob - HIGH_RISK) <= REVIEW_MARGIN
)

print("Invalid enrolled values:", invalid_enrolled)
display(
    enrolled_scores[
        ["Risk Score", "Risk Level", "Human Review"]
    ].head(10)
)

risk_counts = (
    enrolled_scores["Risk Level"]
    .value_counts()
    .reindex(["Low", "Medium", "High"], fill_value=0)
)

risk_counts.plot(
    kind="bar",
    figsize=(6, 4),
    title="Current Risk"
)

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# 19) Explainable AI — SHAP
import shap

pre = best_model.named_steps["preprocessor"]
clf = best_model.named_steps["classifier"]

if isinstance(clf, XGBClassifier):
    transformed_names = pre.get_feature_names_out()

    def raw_name(name):
        if name.startswith("num__"):
            return name.replace("num__", "", 1)

        if name.startswith("cat__"):
            rest = name.replace("cat__", "", 1)
            for col in sorted(categorical_cols, key=len, reverse=True):
                if rest.startswith(col + "_"):
                    return col

        return name

    raw_feature_names = [raw_name(n) for n in transformed_names]

    shap_sample = X_test.sample(
        n=min(300, len(X_test)),
        random_state=RANDOM_STATE
    )

    explainer = shap.TreeExplainer(clf)
    shap_values = explainer.shap_values(
        pre.transform(shap_sample)
    )

    shap_global = (
        pd.DataFrame({
            "Feature": raw_feature_names,
            "Mean |SHAP|": np.abs(shap_values).mean(axis=0)
        })
        .groupby("Feature", as_index=False)["Mean |SHAP|"]
        .sum()
        .sort_values("Mean |SHAP|", ascending=False)
    )

    display(shap_global.head(15))

    highest_idx = enrolled_scores["Risk Score"].idxmax()
    highest_student = enrolled_X.loc[[highest_idx]]

    local_shap = explainer.shap_values(
        pre.transform(highest_student)
    )[0]

    local_explanation = (
        pd.DataFrame({
            "Feature": raw_feature_names,
            "SHAP": local_shap
        })
        .groupby("Feature", as_index=False)["SHAP"]
        .sum()
    )

    local_explanation["Impact"] = local_explanation["SHAP"].abs()
    local_explanation["Direction"] = np.where(
        local_explanation["SHAP"] > 0,
        "Higher risk",
        "Lower risk"
    )

    local_explanation = local_explanation.sort_values(
        "Impact",
        ascending=False
    )

    display(local_explanation.head(10))

else:
    print("SHAP local explanation skipped: selected model is not XGBoost.")
    print("Permutation Importance remains available as global explanation.")


In [ ]:
# 20) Recommendations + Fairness
def advisor_recommendations(row):
    recs = []

    enrolled_units = row["Curricular units 1st sem (enrolled)"]
    approved_units = row["Curricular units 1st sem (approved)"]
    pass_rate = approved_units / enrolled_units if enrolled_units > 0 else 0

    if pass_rate < 0.50:
        recs.append("Academic advisor meeting + tutoring.")

    if row["Curricular units 1st sem (grade)"] < 10:
        recs.append("Academic support plan.")

    if row["Curricular units 1st sem (without evaluations)"] > 0:
        recs.append("Review missed assessments.")

    if row["Tuition fees up to date"] == 0:
        recs.append("Financial/administrative support.")

    if row["Debtor"] == 1:
        recs.append("Review payment-plan options.")

    if not recs:
        recs.append("Routine monitoring.")

    return recs

highest_idx = enrolled_scores["Risk Score"].idxmax()
highest_student = enrolled_X.loc[[highest_idx]]

print("Recommendations:")
for rec in advisor_recommendations(highest_student.iloc[0]):
    print("-", rec)

audit = X_test[["Gender", "International"]].copy()
audit["y_true"] = np.asarray(y_test)
audit["y_pred"] = test_pred

FAIRNESS_LABELS = {
    "Gender": {0: "Female", 1: "Male"},
    "International": {0: "No", 1: "Yes"}
}

def subgroup_metrics(data, col):
    rows = []

    for value, group in data.groupby(col):
        if len(group) < 10:
            continue

        tp = ((group.y_true == 1) & (group.y_pred == 1)).sum()
        fn = ((group.y_true == 1) & (group.y_pred == 0)).sum()
        fp = ((group.y_true == 0) & (group.y_pred == 1)).sum()
        tn = ((group.y_true == 0) & (group.y_pred == 0)).sum()

        rows.append({
            "Group": col,
            "Value": FAIRNESS_LABELS.get(col, {}).get(value, value),
            "Students": len(group),
            "Recall": tp / (tp + fn) if tp + fn else np.nan,
            "False Positive Rate": fp / (fp + tn) if fp + tn else np.nan,
            "Note": "Small subgroup" if len(group) < 30 else ""
        })

    return pd.DataFrame(rows)

fairness_table = pd.concat([
    subgroup_metrics(audit, "Gender"),
    subgroup_metrics(audit, "International")
], ignore_index=True)

display(fairness_table)


In [ ]:
# 21) Flask Export
COURSE_MAP = {
    33: "Biofuel Production Technologies",
    171: "Animation and Multimedia Design",
    8014: "Social Service (evening)",
    9003: "Agronomy",
    9070: "Communication Design",
    9085: "Veterinary Nursing",
    9119: "Informatics Engineering",
    9130: "Equinculture",
    9147: "Management",
    9238: "Social Service",
    9254: "Tourism",
    9500: "Nursing",
    9556: "Oral Hygiene",
    9670: "Advertising and Marketing Management",
    9773: "Journalism and Communication",
    9853: "Basic Education",
    9991: "Management (evening)"
}

BINARY_MAPS = {
    "Daytime/evening attendance": {0: "Evening", 1: "Daytime"},
    "Displaced": {0: "No", 1: "Yes"},
    "Educational special needs": {0: "No", 1: "Yes"},
    "Debtor": {0: "No", 1: "Yes"},
    "Tuition fees up to date": {0: "No", 1: "Yes"},
    "Gender": {0: "Female", 1: "Male"},
    "Scholarship holder": {0: "No", 1: "Yes"},
    "International": {0: "No", 1: "Yes"}
}

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd

EXPORT_ROOT = PROJECT_ROOT / "exports" / "final_v2"
MODELS_DIR = EXPORT_ROOT / "models"
ARTIFACTS_DIR = EXPORT_ROOT / "artifacts"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    best_model,
    MODELS_DIR / "student_risk_model.pkl"
)

joblib.dump(
    calibrator,
    MODELS_DIR / "probability_calibrator.pkl"
)

metadata = {
    "selected_model": best_model_name,
    "decision_threshold": float(HIGH_RISK),
    "medium_risk_threshold": float(MEDIUM_RISK),
    "review_margin": float(REVIEW_MARGIN),
    "prediction_point": "End of Semester 1",
    "target": "Dropout vs Graduate",
    "training_rows": int(len(X_train)),
    "calibration_rows": int(len(X_cal)),
    "test_rows": int(len(X_test)),
    "features": list(X.columns),
    "categorical_features": categorical_cols,
    "excluded_future_features": second_sem_cols,
    "test_metrics": {
        k: float(v)
        for k, v in test_metrics.items()
    }
}

with open(
    MODELS_DIR / "model_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(metadata, f, indent=2)

flask_bundle = {
    "model": best_model,
    "calibrator": calibrator,
    "input_features": list(X.columns),
    "dropout_class_index": list(
        best_model.named_steps["classifier"].classes_
    ).index(1),
    "medium_risk_threshold": float(MEDIUM_RISK),
    "high_risk_threshold": float(HIGH_RISK),
    "review_margin": float(REVIEW_MARGIN),
    "course_map": COURSE_MAP,
    "binary_maps": BINARY_MAPS
}

joblib.dump(
    flask_bundle,
    MODELS_DIR / "student_dropout_flask_bundle.pkl"
)

enrolled_scores.to_csv(
    ARTIFACTS_DIR / "enrolled_students_risk_scores.csv",
    index=False
)

existing_dictionary = PROJECT_ROOT / "artifacts" / "feature_dictionary.csv"

if existing_dictionary.exists():
    shutil.copy2(
        existing_dictionary,
        ARTIFACTS_DIR / "feature_dictionary.csv"
    )
else:
    print("feature_dictionary.csv was not found; existing documented dictionary was not replaced.")

with open(
    ARTIFACTS_DIR / "requirements_ml.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(
        "pandas\nnumpy\nmatplotlib\nscikit-learn\n"
        "xgboost\nshap\njoblib\nflask\n"
    )

print("Exported safely to:", EXPORT_ROOT)
print("Production project files were not overwritten.")


# Final Summary
**EDA ✓ Cleaning ✓ Leakage ✓ Outliers ✓ Correlation ✓ Feature Screening ✓ Imbalance ✓ Model-Specific Scaling ✓ CV ✓ Overfitting Check ✓ Calibration ✓ Final Test ✓ Uncertainty ✓ Feature Importance ✓ Risk Score ✓ Enrolled Scoring ✓ SHAP ✓ Recommendations ✓ Fairness ✓ Safe Flask Export ✓**
